In [1]:
#%load_ext autoreload
#%autoreload 2
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import os
import PIL
import tensorflow as tf
import pathlib

keras = tf.keras
layers = keras.layers
Sequential = keras.Sequential

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.load_metadata import load_metadata
from src.split_dataset import split
from src.input_pipeline import load_image, build_dataset


metadata = load_metadata()

df_train, df_valid, df_test = split(
    metadata,
    training=0.7,
    validation=0.15,
    test=0.15,
    seed=12,
)

unique_char = sorted(metadata["char"].unique())
char_dic = {char: index for index, char in enumerate(unique_char)}
df_train["label"] = df_train["char"].map(char_dic)
df_valid["label"] = df_valid["char"].map(char_dic)
df_test["label"] = df_test["char"].map(char_dic)


batch_size = 32
train_ds = build_dataset(df_train, batch_size=batch_size, training=True)
val_ds = build_dataset(df_valid, batch_size=batch_size, training=False)

In [2]:
train_ds.take(1)

<_TakeDataset element_spec=(TensorSpec(shape=(None, 63, 64, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

In [3]:
model = keras.Sequential([
    layers.Input(shape=(64,63,1)),
    layers.Conv2D(filters=32,kernel_size=(3,3),activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64,3,activation="relu"),
    layers.Flatten(),
    layers.Dense(128,activation="relu"),
    layers.Dense(48,activation="softmax")
])

model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(), 
    metrics=["accuracy"],
)

In [5]:
model.fit(train_ds.take(5), epochs=1)

/Users/nailito/Desktop/Data/katakana-recognition/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.0250 - loss: 3.9342   
